# 01 | Acquire and verify real data

SUPHANATTHAKON THOBSRI | 690631023 | 270702 Data Science Programming

**Scope:** nominal-source-clock-constrained retrospective benchmark. Original historical public release/revision is not certified; operational use remains blocked. Preserve real raw data and never manufacture outcomes.


## Locate the supplied project

Open this notebook inside the extracted project, or upload its complete project ZIP when using Colab. `ROOT` is the project folder; it is not a data-source timestamp. The archive guard rejects paths outside the extraction directory.


In [1]:
from pathlib import Path  # กำหนดตำแหน่งโครงการให้ใช้ได้ในเครื่องและ Colab
import sys  # ใช้โมดูลของโครงการกับ Python ที่กำลังรัน
import os  # อ่านกุญแจจากสภาพแวดล้อมโดยไม่บันทึกค่า
import json  # อ่านค่ากำหนดและข้อมูลอธิบายที่สร้างจริง
import zipfile  # ตรวจและแตกแพ็กเกจโครงการเมื่อจำเป็น
PROJECT_NAME = 'PM25_270702_690631023_Submission'  # ระบุชื่อโครงการฝุ่นและลำดับงานสี่ขั้น
BASE = Path.cwd().resolve()  # ใช้โฟลเดอร์ที่ notebook กำลังรัน
ROOT = BASE if (BASE / 'config.json').is_file() and (BASE / 'src' / 'workflow.py').is_file() else BASE / PROJECT_NAME  # เลือกโครงการที่แตกไฟล์ไว้ก่อน
if not (ROOT / 'src' / 'workflow.py').is_file():  # ค้นหาแพ็กเกจเมื่อเปิด notebook เดี่ยวใน Colab
    if not Path('/content').is_dir():  # การรันในเครื่องต้องมีโครงการที่แตก ZIP อยู่ข้าง notebook
        raise FileNotFoundError('Extract the supplied project ZIP and open its notebook first.')  # หยุดแทนการนำเข้าโค้ดคนละโครงการ
    from google.colab import files  # ขอแพ็กเกจโค้ดและข้อมูลที่มาพร้อม notebook
    uploaded = files.upload()  # อัปโหลด ZIP โครงการที่มีข้อมูลและโค้ดครบ
    archives = [name for name in uploaded if name.lower().endswith('.zip')]  # รับเฉพาะแพ็กเกจ ZIP
    assert len(archives) == 1, 'Upload exactly one project ZIP.'  # หยุดเมื่อเลือกแพ็กเกจได้ไม่แน่นอน
    with zipfile.ZipFile(BASE / archives[0]) as archive:  # เปิด ZIP ที่เลือกเพื่อตรวจความสมบูรณ์
        assert archive.testzip() is None, 'Project ZIP is damaged.'  # ตรวจความสมบูรณ์ ZIP ก่อนแตกไฟล์
        for member in archive.infolist():  # ตรวจปลายทางทุกไฟล์แทนการเชื่อชื่อใน ZIP
            destination = (BASE / member.filename).resolve()  # หาตำแหน่งปลายทางแบบสมบูรณ์ของทุกไฟล์
            assert destination.is_relative_to(BASE), 'Unsafe archive member.'  # ปฏิเสธตำแหน่งไฟล์ที่ออกนอกโฟลเดอร์ทำงาน
        archive.extractall(BASE)  # แตกไฟล์โครงการหลังตรวจความปลอดภัย
assert (ROOT / 'src' / 'workflow.py').is_file(), 'Expected PM2.5 project not found.'  # ตรวจว่ามีโมดูลของโครงการนี้จริง
sys.path.insert(0, str(ROOT / 'src'))  # ใช้โมดูลต้นฉบับที่แนบมากับโครงการ
print('PROJECT:', ROOT)  # แสดงโครงการที่จะรัน

PROJECT: C:\Users\User\Documents\Codex\2026-10-12\computer-assignment-2-2-string-grammar\outputs\PM25_270702_690631023_Submission


## Check dependencies and start a real run

Pinned package versions and the source hashes are recorded. `MODE=replay` validates exact actual API responses already downloaded by the student. Replaying a capture does not imply that it was accessible at every historical issue time. ADS acquisition uses the separate authenticated helper. Public GFS acquisition is available through src/fetch_data.py --acquire-weather NEW_FOLDER; set FETCH_NEW_WEATHER=True only to save a separate capture. No NASA key is requested.


In [2]:
import importlib.util  # ตรวจไลบรารีที่ขาดก่อนนำเข้าโมดูลวิเคราะห์
import subprocess  # ติดตั้งแพ็กเกจด้วย Python ของ notebook เมื่อจำเป็น
REQUIRED_IMPORTS = ['numpy', 'pandas', 'requests', 'matplotlib', 'scipy', 'sklearn', 'nbformat', 'nbclient', 'reportlab', 'pypdf', 'PIL', 'eccodes']  # ระบุไลบรารีที่ต้องใช้
missing_imports = [name for name in REQUIRED_IMPORTS if importlib.util.find_spec(name) is None]  # ตรวจไลบรารีโดยไม่เปลี่ยนแพ็กเกจที่ติดตั้งเงียบ ๆ
if missing_imports:  # ติดตั้งตามรายการรุ่นเมื่อขาดไลบรารีที่จำเป็น
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', str(ROOT / 'requirements.txt')])  # ใช้รุ่นแพ็กเกจที่กำหนดเพื่อให้ทำซ้ำได้
FETCH_NEW_WEATHER = False  # เปิดเฉพาะเมื่อต้องการเก็บคำตอบ API ชุดใหม่แยกจากชุดทดลองเดิม
MODE = 'replay'  # ตรวจและใช้ไฟล์ API จริงที่เก็บไว้ตามแผนที่ล็อก
os.environ['MPLBACKEND'] = 'Agg'  # บันทึกกราฟโดยไม่ต้องใช้หน้าต่างแสดงภาพ
for thread_variable in ['OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS']:  # จำกัดเธรดเพื่อให้โมเดลขนาดเล็กทำซ้ำได้
    os.environ.setdefault(thread_variable, '1')  # เคารพค่าที่ผู้ใช้ตั้งและให้ค่าเริ่มต้นที่ย้ายเครื่องได้
from common import load_config  # อ่านตำแหน่งโครงการและค่าการศึกษาที่กำหนด
os.chdir(ROOT)  # ให้ notebook และสคริปต์อ้างอิงโฟลเดอร์โครงการเดียวกัน
ROOT, CONFIG = load_config()  # สร้างโฟลเดอร์ผลลัพธ์และอ่านค่ากำหนดการศึกษา
assert CONFIG['study_contract'] == 'next_day_0700_thai', 'Wrong forecast contract.'  # ตรวจว่ากำลังใช้ข้อกำหนดเวลาทำนายที่กำหนด
from IPython.display import display, Image, HTML  # แสดงตารางหลักฐานและภาพผลการวิเคราะห์
import pandas as pd  # อ่านข้อมูลรายวันและตารางผลลัพธ์
import numpy as np  # ใช้ตรวจตัวเลขแบบทำซ้ำได้
import workflow as lab  # ใช้ฟังก์ชันควบคุม ตรวจ และจัดแพ็กเกจของโครงการ
run_state = lab.begin_run(ROOT, CONFIG, MODE)  # บันทึก hash โค้ดปัจจุบันก่อนเริ่มวิเคราะห์
display(pd.read_csv(ROOT / 'configuration_dictionary.csv'))  # อธิบายค่ากำหนดทุกตัวและเหตุผลที่ใช้
from environment_check import check_environment  # ตรวจรุ่นที่ติดตั้งกับรายการที่แนบ
dependency_evidence = check_environment(ROOT, strict=True)  # หยุดเมื่อรุ่นแพ็กเกจไม่ตรงแทนการอ้างว่าทำซ้ำได้
print('DEPENDENCIES:', dependency_evidence['status'])  # แสดงผลตรวจสภาพแวดล้อมจริง
display(HTML("<style>.jp-CodeCell .jp-InputArea-editor,.input_area{background:#f2f2f2!important;border:1px solid #ccc}</style>"))  # แสดงพื้นเซลล์โค้ดสีเทาในหน้าที่รองรับ CSS

,variable,value,reason
0,study_contract,"""next_day_0700_thai""",Fixed study identifier for the 07:00 Thai next...
1,start_date,"""2024-01-01""",Use the first of two full calendar study years...
2,end_date,"""2025-12-31""",Use the last full 2025 year of this fixed GFAS...
3,history_calendar_start,"""2023-12-24""",Eight local-day buffers allow the maximum seve...
4,history_calendar_end,"""2025-12-31""",Retain all source days through year end; out-o...
5,latitude,18.7883,Retain the original Chiang Mai receptor; the a...
6,longitude,98.9853,Retain the original Chiang Mai receptor; do no...
7,timezone,"""Asia/Bangkok""",Aggregate PM/weather on Thailand local days; U...
8,operational_issue_time_local,"""07:00""",Approved 07:00 local issue predicts the next l...
9,pm_source,"""CAMS operational forecast archive, fixed 12 U...",Fix forecast run/vintage/lead identity instead...


DEPENDENCIES: PASS


## Lock the question before new scores

Issue **D 07:00 Thailand = D 00 UTC**, target **P(D+1)** over the next local calendar day. Its start/end are 17/41 hours after issue. H1 PM, H2 PM+weather, H3 PM+fire, H4 all; each uses k=3/5/7 complete days. Sources have different day definitions and publication bounds. The learner grids, primary MAE and warning threshold are fixed before fitting.


In [3]:
analysis_plan = json.loads((ROOT / 'analysis_plan.json').read_text(encoding='utf-8'))  # อ่านแผนที่ล็อกก่อนมีคะแนนรอบใหม่
display(pd.DataFrame(analysis_plan['hypotheses'].items(), columns=['Group', 'Question']))  # แสดงคำถามทั้งสี่กลุ่ม
display(pd.DataFrame(json.loads((ROOT / 'report/study_metadata/study_contract.json').read_text(encoding='utf-8')).items(), columns=['Contract item', 'Meaning']))  # แยกขอบเขตย้อนหลังจากการรับรองพร้อมใช้จริง


,Group,Question
0,H1,PM vs persistence
1,H2,PM+weather vs PM
2,H3,PM+fire vs PM
3,H4,all vs H2 and vs H3


,Contract item,Meaning
0,status,LOCKED_BEFORE_NEW_FITTING
1,locked_at_utc,2026-10-21T21:56:06.872313+00:00
2,scope,Retrospective benchmark with documented nomina...
3,primary_eligible_period,Value-complete days 2024-01-28..2025-12-31 min...
4,issue,D 07:00 Asia/Bangkok = D 00 UTC
5,target,"P(D+1), local 00..24; hourly mean at fixed ADS..."
6,PM,"P(d) from run d-1 12 UTC, leads5..28; P(D-1)....."
7,weather,Previous_day1 forecast values aggregated on lo...
8,fire,GFAS UTC mean days D-2..D-(k+1); density-based...
9,nominal_bounds,CAMS12 UTC by22 UTC; GFAS day s by(s+1)07 UTC;...


## Validate actual captured responses

The student downloaded 50 monthly requests. Every raw payload hash and transfer receipt is checked, then every GRIB message is decoded locally. PM is mc/fc/0001, kg/m³, fixed 12 UTC runs and leads 5–52. GFAS is daily UTC analysis with two native parameters. Message order is not trusted. Weather responses retain their original hourly values, including early missing history. This cell may take several minutes because it checks all raw fields. Weather schema, units, field lengths and exact UTC calendar are checked here before preparation; source null values remain missing.


In [4]:
from weather_acquisition import acquire_weather  # ใช้ตัวดึงอากาศจริงชุดเดียวกับสคริปต์
if FETCH_NEW_WEATHER:  # เปิดเฉพาะเมื่อต้องการเก็บข้อมูลใหม่แยกจากชุดทดลองเดิม
    display(acquire_weather(ROOT, ROOT / 'data/raw/weather_new_capture'))  # เก็บคำตอบ API และหลักฐานจริงไว้แยกชุด
raw_folder, raw_manifest, station_reference = lab.acquire_all(ROOT, CONFIG, mode=MODE)  # ตรวจ raw จริงและสร้าง native exports ใหม่จากข้อมูลดิบ
print('RAW RUN:', raw_manifest['run_id'], '| Preserved files:', len(raw_manifest['files']))  # แสดงรอบข้อมูลโดยไม่เปิดเผย credentials
display(pd.DataFrame(json.loads((ROOT / 'outputs/results/raw_decode_audit.json').read_text(encoding='utf-8'))['records']))  # ตรวจจำนวน field และ missing ของคำขอทั้ง 50


Raw source check 1 /50: pm_2023-12


Raw source check 2 /50: pm_2024-01


Raw source check 3 /50: pm_2024-02


Raw source check 4 /50: pm_2024-03


Raw source check 5 /50: pm_2024-04


...

Raw source check 49 /50: gfas_2025-11


Raw source check 50 /50: gfas_2025-12


station_id   reported_at_local  reference_pm25_rolling24h  cams_pm25_mean24_samples  signed_difference_cams_minus_station
       36t 2026-10-16T21:00:00                       13.3                 12.966667                             -0.333333
       35t 2026-10-16T21:00:00                       16.4                 12.966667                             -3.433333


RAW RUN: study_capture_20261021 | Preserved files: 218


,job,messages,missing_grid_values,raw_sha256
0,pm_2023-12,432,0,7067a338bd5f12fa1b97dd617ffd422e4847cb103086a4...
1,pm_2024-01,1488,0,49962014f6dbaee91b2b686a777c3ee88ef5731c799c2a...
2,pm_2024-02,1392,0,dba0eb8905edc9ed6e686ef88086c7aaa279515ffdd934...
3,pm_2024-03,1488,0,4ec8827f79185251bcded4c24630d2d637ae5015d2d2af...
4,pm_2024-04,1440,0,832828887a06a1f91a1b15d8fd7df64dc3c3db3487620b...
5,pm_2024-05,1488,0,6c717f3825feea43d0589ae3f37dfd4cfb49f8f1071a19...
6,pm_2024-06,1440,0,2b15ff017199ea1ac65209f8d03718fe6277a5f2996d94...
7,pm_2024-07,1488,0,1c7d8eaefe74efc4e79906cbaead2b51cdd6c42f9d1677...
8,pm_2024-08,1488,0,b172336b4ae5838bb56c71f6b97493df204f5cad856ffd...
9,pm_2024-09,1440,0,1228eb9a086ce8078ebd001144623acd449677af395bb6...


## Audit provenance and the independent station snapshot

Current download receipts establish current transfer, not original publication. The preserved Air4Thai/Open-Meteo comparison is a contemporary C6 point check outside 2024–2025. It is not an independent historical target for the new ADS model, and its two stations do not create two primary study receptors.


In [5]:
source_inventory = pd.DataFrame(raw_manifest['files'])  # อ่านรายการไฟล์จริงและ hash ใน snapshot
display(source_inventory[['file', 'bytes', 'status']].head(12))  # ตัวอย่างรายการ โดยไฟล์ครบอยู่ใน manifest
display(pd.read_csv(ROOT / 'outputs/results/station_crosscheck.csv'))  # แสดงการเทียบสถานีที่เก็บไว้ตามเวลาและหน่วยจริง


,file,bytes,status
0,ads/acquisition_lock.json,190,success
1,ads/acquisition_summary.json,698,success
2,ads/ads_full_acquisition.py,50839,success
3,ads/download_ledger.csv,2765,success
4,ads/field_metadata.json,32198655,success
5,ads/jobs/gfas_2023-12/job_state.json,1212,success
6,ads/jobs/gfas_2023-12/qc.json,16422,success
7,ads/jobs/gfas_2023-12/raw.grib,90144,success
8,ads/jobs/gfas_2023-12/transfer_receipt.json,1176,success
9,ads/jobs/gfas_2024-01/job_state.json,1652,success


,station_id,primary,station_name,station_latitude,station_longitude,reported_at_local,timezone,unit,reference_pm25_rolling24h,cams_pm25_mean24_samples,signed_difference_cams_minus_station,absolute_difference,cams_one_hour_earlier_window_mean,difference_with_earlier_boundary,matched_hour_count,cams_grid_latitude,cams_grid_longitude,model_window_first_sample,model_window_last_sample
0,36t,True,Yupparaj Wittayalai School,18.790933,98.99000,2026-10-16T21:00:00,Asia/Bangkok,ug/m3,13.3,12.966667,-0.333333,0.333333,13.095833,-0.204167,24,18.800003,99.0,2026-10-15T22:00:00,2026-10-16T21:00:00
1,35t,False,"City Hall, Chiangmai",18.840732,98.96978,2026-10-16T21:00:00,Asia/Bangkok,ug/m3,16.4,12.966667,-3.433333,3.433333,13.095833,-3.304167,24,18.800003,99.0,2026-10-15T22:00:00,2026-10-16T21:00:00


## Raw checkpoint: schema, units, time and source eligibility

The project-owned checker is part of this lab implementation, not an additional instructor file. Technical PASS covers the stated invariants. Historical publication remains UNVERIFIED and the operational guard remains closed. Native zeros and actual missing are separate states.


In [6]:
from check_dataset import check_raw, require_pass  # ใช้ checker ชุดเดียวกับสคริปต์
raw_validation = require_pass(check_raw(ROOT, CONFIG))  # หยุดเมื่อหลักฐาน raw หรือ native exports เปลี่ยน
display(pd.DataFrame(raw_validation['checks']))  # แสดงทั้งข้อที่ผ่านและข้อที่ยังไม่รับรอง
display(pd.DataFrame(json.loads((ROOT / 'outputs/results/source_time_contract.json').read_text(encoding='utf-8'))['sources']))  # แสดงนาฬิกาของแต่ละแหล่งอย่างชัดเจน


,check,status,evidence
0,supported_fetch_script,PASS,"{'path': 'src/fetch_data.py', 'sha256': 'd2d5f..."
1,selected_raw_hashes,PASS,study_capture_20261021
2,native_decoder_source_identity,PASS,Native source identity; raw decoder is part of...
3,fifty_actual_request_coverage,PASS,"50 actual transfers;36,952 native fields; no f..."
4,native_export_field_metadata.json,PASS,data/interim/native_exports/field_metadata.json
5,native_export_pm_hourly.csv,PASS,data/interim/native_exports/pm_hourly.csv
6,native_export_pm_profiles.csv,PASS,data/interim/native_exports/pm_profiles.csv
7,weather_schema_units_calendar_weather_previous...,PASS,8784 expected UTC hours; source nulls preserved.
8,weather_schema_units_calendar_weather_previous...,PASS,8760 expected UTC hours; source nulls preserved.
9,historical_available_at,UNVERIFIED,Source runs/units and nominal release schedule...


,source,product,source_window,nominal_latest_release_relative_to_issue_hours,original_release_observed,evidence_url,nominal_latest_prediction_time_relative_to_issue_hours,prediction_time_is_observed_public_release
0,PM,ADS CAMS operational forecasts; mc/fc/0001,P(D-1)..P(D-k); local days,-26.0,False,https://confluence.ecmwf.int/spaces/CKB/pages/...,NaN,NaN
1,weather,Open-Meteo GFS previous_day1 forecast history,Local daysD-1..D-k; rain interval end can reac...,NaN,False,https://open-meteo.com/en/docs/previous-runs-api,-31.0,False
2,fire,GFAS v1.2 daily analysis,UTC mean daysD-2..D-(k+1),-17.0,False,https://confluence.ecmwf.int/spaces/CKB/pages/...,NaN,NaN
3,provider_reference,"Same ADS PM product, earlier run","Run D-1 at12UTC, leads29..52; targetD+1",-2.0,False,NaN,NaN,NaN
